## ARIMA Model
## Python version 3.11+

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time # To time execution

# Data and Preprocessing
import yfinance as yf
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller

# ARIMA
import pmdarima as pm # For auto_arima

# Visualization
import plotly.graph_objects as go

# Plotting Style Preferences (Optional)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100


## 2. Configuration

In [6]:
# --- User Defined Parameters ---
ticker = "BTC-USD"
start_date = "2017-11-09"
end_date = "2025-01-01"

# Define Train/Test Split Ratio for the *initial* training phase
train_split_ratio = 0.80

# auto_arima parameters 
ARIMA_SEASONAL_PERIOD = 7 # Weekly seasonality for daily data 


## 3. Data Loading and Preparation

In [7]:
print(f"--- Loading Data for {ticker} ---")
try:
    df_full = yf.download(tickers=[ticker], start=start_date, end=end_date, progress=False)
    if df_full.empty:
        raise ValueError(f"No data downloaded for {ticker}.")

    # Select 'Close' column directly
    if 'Close' not in df_full.columns:
         raise ValueError(f"'Close' column not found in downloaded data for {ticker}.")
    df_full = df_full[['Close']].copy()

    df_full = df_full.asfreq('D')
    df_full.ffill(inplace=True) 
    df_full.dropna(inplace=True)
    if df_full.empty:
        raise ValueError(f"Data for {ticker} became empty after processing.")
    print(f"Loaded {len(df_full)} data points for {ticker} from {df_full.index.min()} to {df_full.index.max()}.")
except Exception as e:
    print(f"Original Error: {e}")
    raise ValueError(f"Failed to load or process data for {ticker}. Check symbol and data source.")

--- Loading Data for BTC-USD ---
Loaded 2610 data points for BTC-USD from 2017-11-09 00:00:00 to 2024-12-31 00:00:00.


## 4. Data Splitting

In [8]:
# Split Data into initial training and test sets
n_total = len(df_full)
n_train = int(train_split_ratio * n_total)
n_test = n_total - n_train # n_test determines the number of walk-forward steps

train_data_df = df_full[:n_train]
test_data_df = df_full[n_train:] # This is the period for walk-forward evaluation

print(f"\nInitial Training Data: {n_train} points ({train_data_df.index.min().strftime('%Y-%m-%d')} to {train_data_df.index.max().strftime('%Y-%m-%d')})")
print(f"Test Data (for walk-forward): {n_test} points ({test_data_df.index.min().strftime('%Y-%m-%d')} to {test_data_df.index.max().strftime('%Y-%m-%d')})")



Initial Training Data: 2088 points (2017-11-09 to 2023-07-28)
Test Data (for walk-forward): 522 points (2023-07-29 to 2024-12-31)


## 5. Stationarity Check (on Initial Training Data)

In [9]:
def check_stationarity(timeseries):
    print("\nResults of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=["Test Statistic", "p-value", "#Lags Used", "# Observations Used"],
    )
    for key, value in dftest[4].items():
        dfoutput[f"Critical Value ({key})"] = value
    print(dfoutput.to_string()) # Use to_string for clean print
    if dftest[1] <= 0.05:
        print("=> Conclusion: Data is likely Stationary (reject H0)")
    else:
        print("=> Conclusion: Data is likely Non-Stationary (fail to reject H0)")

print("\n--- Stationarity Check on Initial Training Data ---")
check_stationarity(train_data_df['Close'])


--- Stationarity Check on Initial Training Data ---

Results of Dickey-Fuller Test:
Test Statistic            -1.459481
p-value                    0.553452
#Lags Used                24.000000
# Observations Used     2063.000000
Critical Value (1%)       -3.433524
Critical Value (5%)       -2.862942
Critical Value (10%)      -2.567516
=> Conclusion: Data is likely Non-Stationary (fail to reject H0)


## 6. Initial ARIMA Model Fit (using auto_arima)

In [10]:
print("\n--- Fitting Initial ARIMA Model on Training Data ---")
start_time_initial_train = time.time()

# Use auto_arima to find the best model once on the initial training data
arima_model = pm.auto_arima(train_data_df['Close'],
                           start_p=1, start_q=1,
                           test='adf',        
                           max_p=3, max_q=3,  
                           m=ARIMA_SEASONAL_PERIOD, 
                           start_P=0, seasonal=(ARIMA_SEASONAL_PERIOD > 1),
                           d=None,           
                           D=None,       
                           trace=False,       
                           error_action='ignore',
                           suppress_warnings=True,
                           stepwise=True)

end_time_initial_train = time.time()
print(f"Initial ARIMA training finished in {end_time_initial_train - start_time_initial_train:.2f} seconds.")
print("\n--- Initial Best Model Found ---")
print(arima_model.summary())
print(f"\nBest ARIMA Order: {arima_model.order}")
print(f"Best Seasonal Order: {arima_model.seasonal_order}")


--- Fitting Initial ARIMA Model on Training Data ---
Initial ARIMA training finished in 45.66 seconds.

--- Initial Best Model Found ---
                                      SARIMAX Results                                      
Dep. Variable:                                   y   No. Observations:                 2088
Model:             SARIMAX(0, 1, 0)x(0, 0, [1], 7)   Log Likelihood              -17323.491
Date:                             Tue, 15 Apr 2025   AIC                          34650.982
Time:                                     02:39:33   BIC                          34662.269
Sample:                                 11-09-2017   HQIC                         34655.118
                                      - 07-28-2023                                         
Covariance Type:                               opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------

## 7. Walk-Forward Validation (Rolling Forecast) Loop

In [ ]:
print(f"\n--- Starting ARIMA Walk-Forward Validation for {n_test} steps ---")
start_time_walk_forward = time.time()

arima_walk_forward_predictions = [] # List to store the 1-step ahead predictions

for t in range(n_test):
    # 1. Predict 1-step ahead from the current state of the model
    yhat = arima_model.predict(n_periods=1)[0]
    arima_walk_forward_predictions.append(yhat)

    # 2. Get the actual value for the current step 't' from the test set
    actual_value = test_data_df['Close'].iloc[t]

    # 3. Update the model with the actual observation.
    # This efficiently updates the model's state without a full refit.
    try:
        arima_model.update(actual_value)
    except Exception as e:
        print(f"Warning: ARIMA update failed at step {t+1} with error: {e}. Model state might be stale.")
        # Depending on the error, might need to consider refitting here or stopping.

    # Log progress periodically
    if (t + 1) % 100 == 0:
        print(f"ARIMA Walk-Forward Step {t+1}/{n_test} complete.")

    # --- Full Refitting Point (Computationally Expensive) ---
    # if RETRAIN_FREQUENCY > 0 and (t + 1) % RETRAIN_FREQUENCY == 0:
    #     print(f"\n--- Refitting ARIMA at step {t+1}/{n_test} ---")
    #     current_history = df_full['Close'].iloc[:n_train + t + 1] 
    #     arima_model = pm.auto_arima(current_history, ...) 
    #     print("ARIMA Refitting complete.")
    # --- End Refitting ---


end_time_walk_forward = time.time()
total_walk_forward_time = end_time_walk_forward - start_time_walk_forward
print(f"\nARIMA Walk-Forward finished in {total_walk_forward_time:.2f} seconds.")

# Ensure predictions list is numpy array for metrics
arima_walk_forward_predictions = np.array(arima_walk_forward_predictions)


--- Starting ARIMA Walk-Forward Validation for 522 steps ---
ARIMA Walk-Forward Step 100/522 complete.
ARIMA Walk-Forward Step 200/522 complete.
ARIMA Walk-Forward Step 300/522 complete.
ARIMA Walk-Forward Step 400/522 complete.


## 8. Evaluate Walk-Forward Performance

In [1]:
# Define the evaluation metrics function 
def evaluate_forecast(y_true, y_pred, model_name):
    """Calculates and prints standard evaluation metrics."""
    # Ensure inputs are 1D
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()

    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    mape = mean_absolute_percentage_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    try:
        r2 = r2_score(y_true_flat, y_pred_flat)
    except ValueError: r2 = np.nan

    print(f"\n--- {model_name} Walk-Forward (t+1) Evaluation Results ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"MAPE: {mape:.4%}")
    print(f"R²:   {r2:.4f}")
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# Evaluate against the actual unscaled test data
y_test_actual = test_data_df['Close'].values
if len(y_test_actual) != len(arima_walk_forward_predictions):
    raise ValueError(f"Length mismatch: Actual test data ({len(y_test_actual)}) vs Predictions ({len(arima_walk_forward_predictions)})")

arima_wf_results = evaluate_forecast(y_test_actual, arima_walk_forward_predictions, f"ARIMA ({ticker})")


NameError: name 'test_data_df' is not defined

## 9. Visualize Walk-Forward Results

In [ ]:
print("\n--- Plotting Walk-Forward Forecasts ---")

# Create DataFrame for plotting
results_df_wf = pd.DataFrame({
    'Actual': y_test_actual.flatten(),
    f'ARIMA (t+1)': arima_walk_forward_predictions.flatten()
}, index=test_data_df.index)

fig = go.Figure()
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf['Actual'], mode='lines', name='Actual Price (Test)', line=dict(color='black')))
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf[f'ARIMA (t+1)'], mode='lines', name='ARIMA Walk-Forward (t+1)', line=dict(color='red', dash='dash')))

fig.update_layout(
    title=f'ARIMA Walk-Forward (t+1) Forecast Comparison for {ticker}',
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    legend_title="Data/Model",
    template="plotly_white"
)
fig.show()

## 10. Walk-Forward Evaluation Period Summary

In [ ]:
print(f"\n--- Walk-Forward Evaluation Summary ---")
print(f"Initial Training Data End Date: {train_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Walk-Forward Evaluation Period: {test_data_df.index.min().strftime('%Y-%m-%d')} to {test_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps (Predictions): {n_test}")